# 🏎️ Jev + Gymnasium CarRacing-v3

This Colab lets TypeSafe Jev drive Gymnasium CarRacing-v3.

There is no hidden PID steering controller or expert racing policy. The handcrafted part is only perception: the 96x96 RGB observation is converted into structured road geometry, motion and a visual speed estimate. Jev chooses the driving action.

Gymnasium continuous action is [steering, gas, brake]. Internally this notebook uses [steering, longitudinal], where positive longitudinal means gas and negative means brake, so gas and brake are never mixed.

Create/get your API key at https://typesafe.ai and store it in Colab Secrets as TYPESAFE_API_KEY.

In [ ]:
#@title 1. Install dependencies
!apt-get update -qq
!apt-get install -y -qq swig > /dev/null
!pip -q install "gymnasium[box2d]" imageio imageio-ffmpeg requests pillow

In [ ]:
#@title 2. Imports and configuration
import os, time, json, getpass
from collections import deque
from pathlib import Path
import numpy as np
import requests
import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import Video, Image, display

API_URL = "https://api.typesafe.ai/v1/systemone"
MODEL = "jev-latest"
ENV_ID = "CarRacing-v3"
SEED = 0
MAX_STEPS = 1000
CONTROL_HORIZON = 4

MAX_STEER_DELTA_PER_FRAME = 0.35
MAX_LONGITUDINAL_DELTA_PER_FRAME = 0.30
PROBABILITY_TEMPERATURE = 0.55
TOP_CHOICE_WEIGHT = 0.35
HISTORY_LEN = 10

VIDEO_EVERY = 2
GIF_EVERY = 5
VIDEO_PATH = "/content/jev_car_racing.mp4"
GIF_PATH = "/content/jev_car_racing.gif"
LOG_PATH = "/content/jev_car_racing_log.json"

REQUEST_TIMEOUT_S = 20
MAX_RETRIES = 3
SCAN_ROWS = [73, 66, 59, 52, 45, 38, 31, 24]

def load_typesafe_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
    if not key:
        key = os.environ.get("TYPESAFE_API_KEY")
    if not key:
        key = getpass.getpass("TypeSafe API key from https://typesafe.ai: ").strip()
    if not key:
        raise RuntimeError("No TYPESAFE_API_KEY supplied.")
    return key

TYPESAFE_API_KEY = load_typesafe_key()
print("✓ API key loaded")
print("Environment:", ENV_ID, "| model:", MODEL)

In [ ]:
#@title 3. Pixel perception — no steering controller here
def _segments(indices):
    indices = np.asarray(indices, dtype=np.int32)
    if len(indices) == 0:
        return []
    cuts = np.where(np.diff(indices) > 1)[0] + 1
    return [x for x in np.split(indices, cuts) if len(x)]

def road_mask_from_rgb(frame):
    img = np.asarray(frame, dtype=np.int16)
    mean = img.mean(axis=2)
    spread = img.max(axis=2) - img.min(axis=2)

    # Standard CarRacing rendering: road gray, grass green.
    mask = (spread < 18) & (mean > 65) & (mean < 165)
    mask[82:, :] = False
    return mask

def scan_road(mask):
    H, W = mask.shape
    previous_center = W / 2.0
    centers, widths, valid = [], [], []

    for y in SCAN_ROWS:
        xs = np.where(mask[int(np.clip(y, 0, H-1))])[0]
        segments = _segments(xs)
        if not segments:
            centers.append(0.0)
            widths.append(0.0)
            valid.append(False)
            continue

        candidates = [s for s in segments if len(s) >= 3] or segments
        seg = min(candidates, key=lambda s: abs(float(s.mean()) - previous_center))
        center = float(seg.mean())
        previous_center = center

        offset = (center - (W-1)/2.0) / ((W-1)/2.0)
        centers.append(float(np.clip(offset, -1.0, 1.0)))
        widths.append(float(len(seg) / W))
        valid.append(True)

    return centers, widths, valid

def estimate_dashboard_speed(frame):
    img = np.asarray(frame, dtype=np.uint8)
    H, W, _ = img.shape
    x0 = max(0, int(round(5 * W / 40.0)) - 1)
    x1 = min(W, int(round(6 * W / 40.0)) + 2)
    y0 = int(round(H - 5 * H / 40.0))
    patch = img[y0:H, x0:x1]

    white = (
        (patch[:,:,0] > 215)
        & (patch[:,:,1] > 215)
        & (patch[:,:,2] > 215)
    )
    rows = white.any(axis=1)
    if not rows.any():
        return 0.0

    ys = np.where(rows)[0]
    height_px = float(ys.max() - ys.min() + 1)
    return float(np.clip(height_px / 0.048, 0.0, 120.0))

def extract_visual_state(frame, previous_frame=None):
    mask = road_mask_from_rgb(frame)
    centers, widths, valid = scan_road(mask)
    vc = [c for c, ok in zip(centers, valid) if ok]
    vw = [w for w, ok in zip(widths, valid) if ok]

    near = vc[0] if vc else 0.0
    far = vc[-1] if vc else 0.0

    if previous_frame is None:
        motion = 0.0
    else:
        a = np.asarray(frame, dtype=np.float32)
        b = np.asarray(previous_frame, dtype=np.float32)
        motion = float(np.mean(np.abs(a[:82] - b[:82])) / 255.0)

    return {
        "road_center_offsets_near_to_far": [round(float(x),3) for x in centers],
        "road_width_fraction_near_to_far": [round(float(x),3) for x in widths],
        "scanline_valid": [bool(x) for x in valid],
        "road_visibility": round(float(np.mean(valid)),3),
        "mean_road_width_fraction": round(float(np.mean(vw)) if vw else 0.0,3),
        "near_center_offset": round(float(near),3),
        "far_center_offset": round(float(far),3),
        "curve_delta_far_minus_near": round(float(far-near),3),
        "dashboard_speed_estimate": round(float(estimate_dashboard_speed(frame)),1),
        "frame_motion": round(float(motion),4),
    }

def perception_overlay(frame):
    img = np.asarray(frame).copy()
    mask = road_mask_from_rgb(frame)
    centers, widths, valid = scan_road(mask)
    out = img.copy()
    out[mask] = (0.6*out[mask] + 0.4*np.array([255,255,0])).astype(np.uint8)

    H, W, _ = out.shape
    for y, c, ok in zip(SCAN_ROWS, centers, valid):
        if not ok:
            continue
        x = int(round(c*((W-1)/2.0) + ((W-1)/2.0)))
        x = int(np.clip(x,0,W-1))
        y = int(np.clip(y,0,H-1))
        out[max(0,y-1):min(H,y+2), max(0,x-1):min(W,x+2)] = [255,0,255]
    return out

print("✓ perception ready")

In [ ]:
#@title 4. 49 complete Jev driving actions
STEER_LEVELS = np.array([-0.90,-0.60,-0.30,0.0,0.30,0.60,0.90], dtype=np.float32)
LONG_LEVELS = np.array([-0.80,-0.40,0.0,0.30,0.55,0.80,1.00], dtype=np.float32)

def to_gym_action(control):
    steer = float(np.clip(control[0], -1.0, 1.0))
    longitudinal = float(np.clip(control[1], -1.0, 1.0))
    return np.array([
        steer,
        max(longitudinal, 0.0),
        max(-longitudinal, 0.0),
    ], dtype=np.float32)

def build_candidates():
    candidates, criteria = {}, {}
    idx = 0

    for steer in STEER_LEVELS:
        for longitudinal in LONG_LEVELS:
            key = f"a{idx:02d}"
            control = np.array([steer, longitudinal], dtype=np.float32)
            gym_action = to_gym_action(control)

            candidates[key] = control

            if longitudinal > 0:
                long_text = f"gas {longitudinal:.2f}"
            elif longitudinal < 0:
                long_text = f"brake {-longitudinal:.2f}"
            else:
                long_text = "coast"

            criteria[key] = (
                f"steering {steer:+.2f} (- left, + right); {long_text}; "
                f"Gym [steer,gas,brake]=[{gym_action[0]:+.2f},{gym_action[1]:.2f},{gym_action[2]:.2f}]"
            )
            idx += 1

    return candidates, criteria

CANDIDATES, CRITERIA = build_candidates()
assert len(CANDIDATES) == 49
print("✓", len(CANDIDATES), "actions")

In [ ]:
#@title 5. Jev racing policy
class JevCarRacingPolicy:
    def __init__(self, api_key):
        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        })
        self.history = deque(maxlen=HISTORY_LEN)
        self.current_control = np.array([0.0,0.0], dtype=np.float32)

    def state(self, vision, episode_return, step):
        history = []
        for h in self.history:
            history.append({
                "control": [round(float(x),3) for x in h["control"]],
                "reward_sum": round(float(h["reward_sum"]),3),
                "terminated": bool(h["terminated"]),
                "road_visibility_after": round(float(h["road_visibility_after"]),3),
            })

        return {
            "environment": "Gymnasium CarRacing-v3",
            "objective": (
                "Maximize cumulative reward by following the road and visiting new road tiles. "
                "Avoid leaving the road/playfield. Use throttle on straights and reduce speed for turns."
            ),
            "action_semantics": {
                "steering": "-1 full left, 0 straight, +1 full right",
                "longitudinal": "+1 full gas, 0 coast, -1 strong brake",
            },
            "vision_semantics": {
                "offsets": "negative = road left of image center, positive = road right; ordered near-to-far",
                "curve_delta": "negative = upcoming left bend, positive = upcoming right bend",
            },
            "step": int(step),
            "episode_return": round(float(episode_return),3),
            "vision": vision,
            "current_control": {
                "steering": round(float(self.current_control[0]),3),
                "longitudinal": round(float(self.current_control[1]),3),
            },
            "recent_action_outcomes": history,
        }

    def call(self, state):
        payload = {
            "model": MODEL,
            "state": state,
            "questions": {
                "driving_action": {
                    "type": "choice",
                    "instructions": (
                        "Choose the complete action most likely to maximize future CarRacing reward. "
                        "Use near-to-far road center offsets to infer the upcoming turn. "
                        "Use road width, visibility, motion, speed estimate and recent reward outcomes. "
                        "Avoid excessive throttle in strong turns and react when road geometry changes."
                    ),
                    "criteria": CRITERIA,
                }
            },
        }

        last_error = None
        for attempt in range(MAX_RETRIES):
            t0 = time.perf_counter()
            try:
                r = self.session.post(API_URL, json=payload, timeout=REQUEST_TIMEOUT_S)
                latency = (time.perf_counter()-t0)*1000.0

                if r.status_code in (429,529) or 500 <= r.status_code < 600:
                    last_error = RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
                    time.sleep(min(2**attempt,4))
                    continue

                r.raise_for_status()
                data = r.json()
                a = data["answers"]["driving_action"]

                return {
                    "choice": a["choice"],
                    "confidence": float(a.get("confidence",0.0)),
                    "probabilities": a.get("probabilities",{}),
                    "latency_ms": latency,
                }
            except Exception as e:
                last_error = e
                if attempt + 1 < MAX_RETRIES:
                    time.sleep(min(2**attempt,4))

        raise RuntimeError(f"Jev request failed: {last_error}")

    def probabilities_to_control(self, d):
        keys, p = [], []
        for key in CANDIDATES:
            v = float(d["probabilities"].get(key,0.0))
            if v > 0:
                keys.append(key)
                p.append(v)

        if not p:
            winner = d["choice"]
            if winner not in CANDIDATES:
                raise RuntimeError(f"Unknown Jev action: {winner}")
            return CANDIDATES[winner].copy(), winner, [(winner,1.0)]

        p = np.asarray(p,dtype=np.float64)
        p = np.maximum(p,1e-12)
        p = p ** (1.0/PROBABILITY_TEMPERATURE)
        p /= p.sum()

        expected = np.zeros(2,dtype=np.float64)
        for key,w in zip(keys,p):
            expected += w*CANDIDATES[key]

        winner = d["choice"] if d["choice"] in CANDIDATES else keys[int(np.argmax(p))]
        target = (
            (1.0-TOP_CHOICE_WEIGHT)*expected
            + TOP_CHOICE_WEIGHT*CANDIDATES[winner]
        )
        target = np.clip(target,-1.0,1.0).astype(np.float32)

        top = sorted(zip(keys,p.tolist()), key=lambda x:x[1], reverse=True)[:5]
        return target, winner, top

    def decide(self, vision, episode_return, step):
        d = self.call(self.state(vision, episode_return, step))
        target, winner, top = self.probabilities_to_control(d)
        return {**d, "target_control":target, "winner":winner, "top_distribution":top}

    def record(self, control, rewards, terminated, vision_after):
        self.history.append({
            "control": np.asarray(control,dtype=np.float32).copy(),
            "reward_sum": float(np.sum(rewards)),
            "terminated": bool(terminated),
            "road_visibility_after": float(vision_after["road_visibility"]),
        })

print("✓ Jev policy ready")

In [ ]:
#@title 6. Sanity-check perception and TypeSafe API
env = gym.make(
    ENV_ID,
    render_mode="rgb_array",
    continuous=True,
    domain_randomize=False,
)
obs, info = env.reset(seed=SEED)

vision = extract_visual_state(obs)
print(json.dumps(vision, indent=2))

plt.figure(figsize=(5,5))
plt.imshow(perception_overlay(obs))
plt.title("Yellow road mask; magenta sampled centerline")
plt.axis("off")
plt.show()

test_policy = JevCarRacingPolicy(TYPESAFE_API_KEY)
test = test_policy.decide(vision, 0.0, 0)

print("✓ Jev API works")
print("confidence:", round(test["confidence"],3))
print("latency_ms:", round(test["latency_ms"],1))
print("target [steer,long]:", np.round(test["target_control"],3).tolist())
print("Gym [steer,gas,brake]:", np.round(to_gym_action(test["target_control"]),3).tolist())
print("top distribution:", test["top_distribution"])

env.close()

In [ ]:
#@title 7. Run Jev-controlled CarRacing
env = gym.make(
    ENV_ID,
    render_mode="rgb_array",
    continuous=True,
    domain_randomize=False,
)
obs, info = env.reset(seed=SEED)
policy = JevCarRacingPolicy(TYPESAFE_API_KEY)

previous_frame = None
episode_return = 0.0
step = 0
decision_log = []
reward_history = []
steer_history = []
long_history = []
visibility_history = []
speed_history = []
confidence_history = []
api_failures = 0
gif_frames = []

writer = imageio.get_writer(
    VIDEO_PATH,
    format="FFMPEG",
    mode="I",
    fps=max(1,50//VIDEO_EVERY),
    codec="libx264",
    pixelformat="yuv420p",
    macro_block_size=1,
)

try:
    while step < MAX_STEPS:
        vision_before = extract_visual_state(obs, previous_frame)

        try:
            d = policy.decide(vision_before, episode_return, step)
            target = d["target_control"].copy()
        except Exception as e:
            api_failures += 1
            target = (0.7*policy.current_control).astype(np.float32)
            d = {
                "choice":"api_fallback",
                "winner":"api_fallback",
                "confidence":0.0,
                "latency_ms":float("nan"),
                "top_distribution":[],
                "target_control":target,
                "error":repr(e),
            }

        start = policy.current_control.copy()
        rewards = []
        terminated = truncated = False

        for substep in range(CONTROL_HORIZON):
            f = float(substep+1)/CONTROL_HORIZON
            desired = (1.0-f)*start + f*target
            delta = desired - policy.current_control

            delta[0] = np.clip(delta[0], -MAX_STEER_DELTA_PER_FRAME, MAX_STEER_DELTA_PER_FRAME)
            delta[1] = np.clip(delta[1], -MAX_LONGITUDINAL_DELTA_PER_FRAME, MAX_LONGITUDINAL_DELTA_PER_FRAME)

            control = np.clip(policy.current_control + delta, -1.0, 1.0).astype(np.float32)
            gym_action = to_gym_action(control)

            old_obs = obs
            obs, reward, terminated, truncated, info = env.step(gym_action)
            previous_frame = old_obs
            policy.current_control = control

            episode_return += float(reward)
            rewards.append(float(reward))
            step += 1

            v = extract_visual_state(obs, previous_frame)
            reward_history.append(episode_return)
            steer_history.append(float(control[0]))
            long_history.append(float(control[1]))
            visibility_history.append(float(v["road_visibility"]))
            speed_history.append(float(v["dashboard_speed_estimate"]))

            if step % VIDEO_EVERY == 0:
                writer.append_data(env.render())

            if step % GIF_EVERY == 0:
                frame = env.render()
                gif_frames.append(np.asarray(PILImage.fromarray(frame).resize((480,384))))

            if terminated or truncated or step >= MAX_STEPS:
                break

        vision_after = extract_visual_state(obs, previous_frame)
        policy.record(target, rewards, terminated or truncated, vision_after)

        decision_log.append({
            "step":step,
            "choice":d["choice"],
            "winner":d["winner"],
            "confidence":float(d["confidence"]),
            "latency_ms":float(d["latency_ms"]),
            "target_control":[float(x) for x in target],
            "macro_reward":float(np.sum(rewards)),
            "episode_return":float(episode_return),
            "vision_before":vision_before,
            "vision_after":vision_after,
            "top_distribution":d["top_distribution"],
        })
        confidence_history.append(float(d["confidence"]))

        if len(decision_log) <= 12 or len(decision_log)%20 == 0 or terminated or truncated:
            print(
                f"decision {len(decision_log):04d} | step {step:04d} | "
                f"conf {d['confidence']:.3f} | {d['latency_ms']:.0f} ms | "
                f"reward {np.sum(rewards):+.2f} | return {episode_return:+.1f} | "
                f"steer {target[0]:+.2f} | long {target[1]:+.2f} | "
                f"near {vision_before['near_center_offset']:+.2f} | "
                f"far {vision_before['far_center_offset']:+.2f} | "
                f"curve {vision_before['curve_delta_far_minus_near']:+.2f} | "
                f"speed~ {vision_before['dashboard_speed_estimate']:.0f}"
            )

        if terminated or truncated:
            print("Episode ended at", step, "terminated=",terminated,"truncated=",truncated)
            break
finally:
    writer.close()
    env.close()

if gif_frames:
    imageio.mimsave(GIF_PATH, gif_frames, duration=GIF_EVERY/50.0, loop=0)

Path(LOG_PATH).write_text(
    json.dumps({
        "environment":ENV_ID,
        "model":MODEL,
        "decisions":decision_log,
    }, indent=2),
    encoding="utf-8",
)

print("\n=== RESULT ===")
print("steps:",step)
print("total reward:",round(episode_return,2))
print("Jev decisions:",len(decision_log))
print("API failures:",api_failures)

lat = [x["latency_ms"] for x in decision_log if np.isfinite(x["latency_ms"])]
if lat:
    print("mean latency ms:",round(float(np.mean(lat)),1))
    print("p95 latency ms:",round(float(np.percentile(lat,95)),1))

In [ ]:
#@title 8. Video / GIF / plots
print("MP4:")
display(Video(VIDEO_PATH, embed=True, html_attributes="controls loop"))

print("GIF fallback:")
display(Image(filename=GIF_PATH))

plt.figure(figsize=(12,4))
plt.plot(reward_history)
plt.xlabel("Gym step")
plt.ylabel("cumulative reward")
plt.title("Jev CarRacing reward")
plt.grid(True,alpha=.25)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(steer_history,label="steering")
plt.plot(long_history,label="longitudinal (+gas / -brake)")
plt.ylim(-1.05,1.05)
plt.legend()
plt.grid(True,alpha=.25)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(visibility_history,label="road visibility")
plt.plot(np.asarray(speed_history)/120.0,label="speed estimate / 120")
plt.legend()
plt.grid(True,alpha=.25)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(confidence_history)
plt.ylim(-.02,1.02)
plt.title("Jev action confidence")
plt.grid(True,alpha=.25)
plt.show()

print("MP4:",VIDEO_PATH)
print("GIF:",GIF_PATH)
print("log:",LOG_PATH)